# 03 - Preprocessing Service Test

هذا الـ Notebook يختبر خدمة المعالجة فقط:
- `EmbeddingTextCleaner`
- `QueryProcessor`
- `Tokenizer`

لا نعيد كتابة خوارزمية preprocessing داخل Notebook.

In [ ]:
from pathlib import Path
import sys, os, json, sqlite3, subprocess, time
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# يستطيع صديقك تغيير هذا المتغير إذا كانت artifacts خارج المشروع.
# مثال في PowerShell قبل فتح notebook:
# $env:IR_ARTIFACT_ROOT="E:\\ir_project_artifacts"
ARTIFACT_ROOT = Path(os.environ.get("IR_ARTIFACT_ROOT", r"E:\ir_project_artifacts"))


def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return Path(paths[0])

DB_PATH = first_existing([
    PROJECT_ROOT / "data" / "documents.sqlite",
    PROJECT_ROOT / "data" / "documents.db",
    ARTIFACT_ROOT / "documents.sqlite",
    ARTIFACT_ROOT / "documents.db",
])

TERRIER_INDEX_PATH = first_existing([
    PROJECT_ROOT / "indexes" / "terrier_medline",
    PROJECT_ROOT / "data" / "indexes" / "terrier_medline",
    ARTIFACT_ROOT / "indexes" / "terrier_medline",
])

BERT_INDEX_DIR = first_existing([
    PROJECT_ROOT / "indexes" / "faiss_bert",
    PROJECT_ROOT / "indexes" / "faiss_bert_full",
    PROJECT_ROOT / "data" / "rag_artifacts",
    ARTIFACT_ROOT / "indexes" / "faiss_bert_full",
    ARTIFACT_ROOT / "indexes" / "faiss_bert",
])

WORD2VEC_INDEX_DIR = first_existing([
    PROJECT_ROOT / "indexes" / "faiss_word2vec",
    PROJECT_ROOT / "indexes" / "faiss_word2vec_full",
    ARTIFACT_ROOT / "indexes" / "faiss_word2vec_full",
    ARTIFACT_ROOT / "indexes" / "faiss_word2vec",
])

REPORTS_DIR = PROJECT_ROOT / "reports" / "evaluation_notebook"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT       =", PROJECT_ROOT)
print("ARTIFACT_ROOT      =", ARTIFACT_ROOT)
print("DB_PATH            =", DB_PATH, "exists=", DB_PATH.exists())
print("TERRIER_INDEX_PATH =", TERRIER_INDEX_PATH, "exists=", TERRIER_INDEX_PATH.exists())
print("BERT_INDEX_DIR     =", BERT_INDEX_DIR, "exists=", BERT_INDEX_DIR.exists())
print("WORD2VEC_INDEX_DIR =", WORD2VEC_INDEX_DIR, "exists=", WORD2VEC_INDEX_DIR.exists())

In [ ]:
from src.preprocessing import EmbeddingTextCleaner, QueryProcessor
from src.preprocessing.tokenizer import Tokenizer

cleaner = EmbeddingTextCleaner(max_chars=None)
query_processor = QueryProcessor()
tokenizer = Tokenizer()

print("Preprocessing services loaded.")

In [ ]:
test_texts = [
    "BRCA1 and BRCA2 mutations are associated with breast cancer.",
    "Gene-expression in tumor cells: p53, TNF-alpha, and IL-2.",
    "This document contains HTML <b>tags</b> and punctuation!!!",
]

rows = []
for text in test_texts:
    cleaned = cleaner.clean(text)
    tokens = tokenizer.tokenize(cleaned)
    rows.append({"original": text, "cleaned": cleaned, "tokens": tokens})

pd.DataFrame(rows)

In [ ]:
queries = [
    "What genes are associated with breast cancer?",
    "BRCA1 mutation breast cancer",
    "tumor protein p53 gene expression",
]

rows = []
for q in queries:
    processed = query_processor.process(q)
    rows.append({"query": q, "processed_query": processed})

pd.DataFrame(rows)